# Validate the Azure ML Workshop

This read-only notebook validates the cloned workshop, the single `.env`, local tools, Azure authentication, workspace access, compute access, and representative predeployed assets.

## Sources

The structure follows this repository's Azure ML SDK v2 notebooks and the [Azure ML examples repository](https://github.com/Azure/azureml-examples). See `workshop/SOURCES.md` for the source and license ledger.

In [1]:
# 1. Import Required Libraries
from itertools import islice
from pathlib import Path
import os
import shutil
import sys

import pandas as pd
from azure.ai.ml import MLClient
from azure.core.exceptions import HttpResponseError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

In [2]:
# 2. Define Configuration
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (
        (candidate / ".env.example").is_file()
        and (candidate / "pipelines").is_dir()
        and (candidate / "notebooks").is_dir()
    ):
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

ENV_FILE = WORKSHOP_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError("Copy workshop/.env.example to workshop/.env and provide workspace values")
load_dotenv(ENV_FILE, override=True)

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "").strip()
TENANT_ID = os.getenv("AZURE_TENANT_ID", "").strip()
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "").strip()
WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "").strip()
COMPUTE_NAME = os.getenv("AZUREML_COMPUTE_NAME", "").strip()
COMPUTE_IDENTITY_CLIENT_ID = os.getenv("AZUREML_COMPUTE_IDENTITY_CLIENT_ID", "").strip()
INSTANCE_IDENTITY_CLIENT_ID = os.getenv("DEFAULT_IDENTITY_CLIENT_ID", "").strip()
H2O_VERSION = os.getenv("H2O_VERSION", "").strip()

required = {
    "AZURE_SUBSCRIPTION_ID": SUBSCRIPTION_ID,
    "AZURE_RESOURCE_GROUP": RESOURCE_GROUP,
    "AZUREML_WORKSPACE_NAME": WORKSPACE_NAME,
    "AZUREML_COMPUTE_NAME": COMPUTE_NAME,
    "AZUREML_COMPUTE_IDENTITY_CLIENT_ID": COMPUTE_IDENTITY_CLIENT_ID,
    "H2O_VERSION": H2O_VERSION,
}
missing = [name for name, value in required.items() if not value or value.startswith("<")]
if missing:
    raise ValueError("Missing or placeholder .env values: " + ", ".join(missing))
if not INSTANCE_IDENTITY_CLIENT_ID:
    raise EnvironmentError("DEFAULT_IDENTITY_CLIENT_ID is not set; assign a UMI to this compute instance")

credential = AzureCliCredential(tenant_id=TENANT_ID or None)
access_token = credential.get_token("https://management.azure.com/.default")

import jwt

token_claims = jwt.decode(access_token.token, options={"verify_signature": False})
cli_identity_client_id = token_claims.get("appid") or token_claims.get("azp")
if cli_identity_client_id != INSTANCE_IDENTITY_CLIENT_ID:
    raise PermissionError(
        "Azure CLI is not logged in as the compute-instance UMI. Run: "
        "az login --identity --client-id $DEFAULT_IDENTITY_CLIENT_ID"
    )

ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)

In [3]:
# 3. Implement Core Functions
def command_available(name: str) -> bool:
    """Return whether a command is available on PATH."""
    if not name.strip():
        raise ValueError("Command name cannot be empty")
    return shutil.which(name) is not None


def first_names(items, limit: int = 5) -> list[str]:
    """Return up to limit asset names without exhausting a paged Azure iterator."""
    if limit < 1:
        raise ValueError("Limit must be positive")
    return [item.name for item in islice(items, limit)]


def add_rows(rows: list[dict], asset_type: str, names: list[str]) -> None:
    """Append display rows for a named Azure ML asset collection."""
    rows.extend({"asset_type": asset_type, "name": name} for name in names)

In [4]:
# 4. Run the Implementation
workspace_scope = (
    f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
)
identity_object_id = token_claims.get("oid", "<UMI_OBJECT_ID>")

try:
    workspace = ml_client.workspaces.get(WORKSPACE_NAME)
    compute = ml_client.compute.get(COMPUTE_NAME)
except HttpResponseError as error:
    if error.status_code == 403:
        raise PermissionError(
            "The compute-instance UMI authenticated successfully but cannot read the Azure ML workspace. "
            "An Azure RBAC administrator must run:\n"
            f"az role assignment create --assignee-object-id {identity_object_id} "
            "--assignee-principal-type ServicePrincipal --role 'AzureML Data Scientist' "
            f"--scope '{workspace_scope}'\n"
            "After role propagation, run "
            "az login --identity --client-id $DEFAULT_IDENTITY_CLIENT_ID and rerun Cells 3-6."
        ) from error
    raise

cluster_identities = getattr(getattr(compute, "identity", None), "user_assigned_identities", None) or []
cluster_identity_client_ids = {
    identity.client_id for identity in cluster_identities if getattr(identity, "client_id", None)
}
if COMPUTE_IDENTITY_CLIENT_ID not in cluster_identity_client_ids:
    raise PermissionError(
        f"Compute {COMPUTE_NAME} does not expose the configured cluster UMI client ID "
        f"{COMPUTE_IDENTITY_CLIENT_ID}"
    )

print(f"Workspace: {workspace.name}")
print(f"Resource group: {workspace.resource_group}")
print(f"Compute: {compute.name} ({compute.type})")
print(f"Compute-instance UMI client ID: {INSTANCE_IDENTITY_CLIENT_ID}")
print(f"Compute-cluster UMI client ID: {COMPUTE_IDENTITY_CLIENT_ID}")
print(f"Python: {sys.version.split()[0]}")
print({name: command_available(name) for name in ("az", "git", "java", "uv")})

rows = []
add_rows(rows, "data", first_names(ml_client.data.list()))
add_rows(rows, "model", first_names(ml_client.models.list()))
add_rows(rows, "environment", first_names(ml_client.environments.list()))
add_rows(rows, "job", first_names(ml_client.jobs.list()))
add_rows(rows, "endpoint", first_names(ml_client.online_endpoints.list()))
asset_summary = pd.DataFrame(rows)
display(asset_summary)

Workspace: mlwdevcc01
Resource group: rg-aml-ws-dev-cc-01
Compute: aml-cluster-dev-cc01 (amlcompute)
Compute-instance UMI client ID: 1ae9b3dc-5b0f-4cd8-bca4-a9180c160f9f
Compute-cluster UMI client ID: 30fd8338-79f3-4fff-b959-c56d11672e31
Python: 3.12.14
{'az': True, 'git': True, 'java': True, 'uv': True}


,asset_type,name
0,data,taxi-h2o-batch-input
1,data,cat-sample-test-data
2,data,cat-sample-train-data
3,data,workshop-taxi-data
4,data,workshop-taxi-endpoint-collector-model_inputs
5,model,taxi-fare-h2o-binary
6,model,azureml_tender_carnival_ty17ndxss4_0_output_ml...
7,model,workshop-taxi-model
8,model,azureml_coral_oven_z46ykf5lyj_0_output_mlflow_...
9,model,workshop-h2o-binary


In [5]:
# 5. Validate Expected Behavior
assert WORKSHOP_ROOT.name == "workshop"
assert workspace.name == WORKSPACE_NAME
assert compute.name == COMPUTE_NAME
assert (WORKSHOP_ROOT / "pipelines/single-step-merge-job.yaml").is_file()
assert (WORKSHOP_ROOT / "pipelines/integration-compare-pipeline.yaml").is_file()
assert command_available("az")
assert command_available("git")
assert command_available("java")
print("Workshop preflight passed. No Azure resources were changed.")

Workshop preflight passed. No Azure resources were changed.


## Expected Result and Next Step

The notebook reports the configured workspace and compute, confirms required local tools, and shows representative predeployed assets without changing Azure.

Next: `notebooks/01_foundations/01_workspace_and_compute.ipynb`.